In [40]:
import io
import base64
import segno
from PIL import Image

# Read the file
with open("./src.jpg", "rb") as f:
    data = f.read()

# 800 raw bytes becomes ~1066 Base64 chars. Safe for fast scanning.
CHUNK = 800  

chunks = [data[i:i + CHUNK] for i in range(0, len(data), CHUNK)]
total = len(chunks)

frames = []
durations = []

print(f"Generating {total} Base64 QR codes...")

for idx, chunk in enumerate(chunks, start=1):
    # Encode chunk to Base64 safely
    b64_chunk = base64.b64encode(chunk).decode("ascii")
    payload = f"{idx}/{total}|{b64_chunk}"
    
    # Keep error='l' (Low) so the QR grid is simple and scans instantly
    qr = segno.make(payload, error="l", micro=False)

    buffer = io.BytesIO()
    # Keep scale low for fast GIF generation
    qr.save(buffer, kind="png", scale=5, border=2, dark="black", light="white")
    buffer.seek(0)

    qr_img = Image.open(buffer)
    frames.append(qr_img)
    durations.append(100) # 100ms QR flash

    # white_frame = Image.new("RGB", qr_img.size, (255, 255, 255))
    # frames.append(white_frame)
    # durations.append(20)  # 20ms visual delimiter

print("Saving optimized looping GIF...")
frames[0].save(
    "qr_transfer.gif",
    save_all=True,
    append_images=frames[1:],
    duration=durations,
    loop=0,
    optimize=False,     
    compress_level=1    
)

print("Done! Open qr_transfer.gif")

Generating 176 Base64 QR codes...
Saving optimized looping GIF...
Done! Open qr_transfer.gif


In [30]:
import os
import re
import cv2
import cairosvg
import numpy as np

In [26]:
QR_DIR = "./one"
OUT_VIDEO = "qr_transfer.mp4"

FPS = 30              # frames per second
FRAMES_PER_QR = 9    # repeat each QR so scanners can lock on
RES = 1080            # output video is RES x RES (square)
PAD_RATIO = 0.10      # white margin around each QR

In [27]:
# Collect and sort by numeric index so 1,2,...,10 (not 1,10,2)
files = [f for f in os.listdir(QR_DIR) if f.lower().endswith(".svg")]
files.sort(key=lambda f: int(re.search(r"\d+", f).group()))

print(f"Found {len(files)} QR files")

Found 141 QR files


In [28]:
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
vw = cv2.VideoWriter(OUT_VIDEO, fourcc, FPS, (RES, RES))

In [31]:
for idx, fname in enumerate(files, 1):
    svg_path = os.path.join(QR_DIR, fname)

    # Render SVG → PNG in memory at high resolution
    png_bytes = cairosvg.svg2png(url=svg_path, output_width=1000, output_height=1000)
    arr = np.frombuffer(png_bytes, dtype=np.uint8)
    img = cv2.imdecode(arr, cv2.IMREAD_GRAYSCALE)

    # Pad to square with white (in case it isn't already)
    h, w = img.shape
    side = max(h, w)
    canvas = np.full((side, side), 255, dtype=np.uint8)
    y0, x0 = (side - h) // 2, (side - w) // 2
    canvas[y0:y0+h, x0:x0+w] = img

    # Add white quiet-zone border
    pad = int(side * PAD_RATIO)
    framed = np.full((side + 2*pad, side + 2*pad), 255, dtype=np.uint8)
    framed[pad:pad+side, pad:pad+side] = canvas

    # Resize to video resolution
    frame = cv2.resize(framed, (RES, RES), interpolation=cv2.INTER_NEAREST)

    # Write FRAMES_PER_QR identical frames
    for _ in range(FRAMES_PER_QR):
        vw.write(cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR))

    # Write 1 pure white frame as a visual delimiter (~33ms pause)
    white_frame = np.full((RES, RES, 3), 255, dtype=np.uint8)
    vw.write(white_frame)

    if idx % 10 == 0 or idx == len(files):
        print(f"Wrote {idx}/{len(files)}")

vw.release()
print(f"Done → {OUT_VIDEO}")

Wrote 10/141
Wrote 20/141
Wrote 30/141
Wrote 40/141
Wrote 50/141
Wrote 60/141
Wrote 70/141
Wrote 80/141
Wrote 90/141
Wrote 100/141
Wrote 110/141
Wrote 120/141
Wrote 130/141
Wrote 140/141
Wrote 141/141
Done → qr_transfer.mp4
